In [1]:
import sys
import pandas as pd
sys.path.insert(0, '../../')

In [2]:
from src.permutation_test import find_paired_permutation_test
from src.outlier_detection import find_outliers_remove
from sklearn.model_selection import train_test_split

## Data Preparation

In [4]:
remove_zero = True

# Load Extracted features dataset
ML6_1 = pd.read_excel('../../dataset/ML6/Day 1 Data/feature_extraction_noise_None.xlsx')
ML6_2 = pd.read_excel('../../dataset/ML6/Day 2 Data/feature_extraction_noise_None.xlsx')

ML6_1['dataset_name'] = '1'
ML6_2['dataset_name'] = '2'

final_dataset = pd.concat([ML6_1, ML6_2], ignore_index=True)
final_dataset['label'] = final_dataset['file'].apply(lambda x: x.split('/')[-1].split('_')[-2].replace('cbz','')).apply(lambda x: int(x))
final_dataset.rename(columns={"PH": 'max(S)', 'signal_std':'std(S)', 'signal_mean':'mean(S)', 'peak area':'area(S)', \
                        'dS_dV_area':'area(dS/dV)', 'dS_dV_max_peak':'max(dS/dV)', 'dS_dV_min_peak':'min(dS/dV)',\
                    'dS_dV_peak_diff':'max(dS/dV) - min(dS/dV)', \
                    'peak V':'V_max(S)', 'dS_dV_max_V':'V_max(dS/dV)', 'dS_dV_min_V':'V_min(dS/dV)',\
        }, inplace = True)

final_dataset = final_dataset[['area(S)', 'max(S)', 'max(dS/dV)', 'min(dS/dV)', 'V_max(S)', 'vcenter', 'V_max(dS/dV)', 'V_min(dS/dV)', 'f1', 'f2', 'file', 'label', 'dataset_name']]

# Split dataset to train test
train, test  = train_test_split(final_dataset, test_size=0.4, shuffle=True, random_state=42, stratify=final_dataset['label'])

# Remove outliers with feature 'max(S)'
X_filtered, outlier_S = find_outliers_remove('max(S)',  train)

# Remove outliers with feature 'V_max(S)'
X_filtered, outlier_V = find_outliers_remove('V_max(S)', X_filtered)

all_outliers = [ x for i in outlier_S for x in (outlier_S[i]['above'] + outlier_S[i]['below'])] + \
               [ x for i in outlier_V for x in (outlier_V[i]['above'] + outlier_V[i]['below'])]

# Check if the outliers are in the dataframe or not
assert (not(X_filtered['file'].apply(lambda x: x.split('/')[-1]).isin(all_outliers).any()))

X_train = X_filtered.drop(['label', 'file', 'dataset_name'], axis=1)
y_train = X_filtered['label']
X_test  = test.drop(['label', 'file', 'dataset_name'], axis=1)
y_test  = test['label']

# Remove zero only from testing dataset
if remove_zero:
    indx_gt_zero   = list(y_test[y_test!=0].index)
    X_test         = X_test.loc[indx_gt_zero]
    y_test         = y_test.loc[indx_gt_zero]

In [5]:
features_list= {'KNN':    ['min(dS/dV)', 'max(S)'], 
                'Linear': ['min(dS/dV)', 'V_max(S)', 'f2', 'f1', 'V_max(dS/dV)'], 
                'RF':     ['min(dS/dV)', 'area(S)', 'f2', 'V_max(dS/dV)'],
                'SVM':    ['min(dS/dV)', 'area(S)', 'vcenter', 'V_max(dS/dV)', 'max(dS/dV)'], 
                'GP':     ['min(dS/dV)', 'area(S)', 'V_max(dS/dV)', 'max(dS/dV)']}

paired_test = [('KNN', 'Linear'),
               ('KNN', 'SVM'), 
               ('KNN', 'GP'),
               ('KNN', 'RF'),
               ('RF',  'Linear'),
               ('RF',  'SVM'),
               ('RF',  'GP'),
              ]

In [7]:
dataset     = (X_train, X_test, y_train, y_test)
output      = find_paired_permutation_test(dataset, features_list, paired_test)

  0%|                                                                                                      | 0/7 [00:00<?, ?it/s]/nfs/hpc/share/buddhacs/Epilepsy/vgramreg/notebooks/Journal_Paper/../../src/permutation_test.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, temp.dropna(axis=1, how='all')], ignore_index=True)
100%|██████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [02:15<00:00, 19.39s/it]


In [8]:
output

,Model Comparison,Observed Diff,Diff mean,Diff std,p value
0,KNN--------- Linear,3.872534,2.135365,1.598953,0.1471
1,KNN--------- SVM,-0.237277,1.546532,1.143414,0.9028
2,KNN--------- GP,-0.120247,1.344267,0.995375,0.9393
3,KNN--------- RF,4.966401,1.391386,1.021829,0.0017
4,RF--------- Linear,-1.257010,1.808707,1.354454,0.5793
5,RF--------- SVM,-5.404766,1.607331,1.202150,0.0054
6,RF--------- GP,-5.424796,1.426321,1.057586,0.0016


In [10]:
output.to_excel('../../results/Journal_paper/ML6_zero_removal/paired_permutation_test.xlsx')